# RL agent training on Kaggle (free GPU)

Trains the PPO agent from this repository with a large budget and records the result in the
project's model registry (Supabase). The registry runs the same champion/challenger check as
the nightly job: the new version trades on the demo account only if it beats the current
champion on the holdout Sortino ratio.

**One-time setup (free, no card):**
1. Notebook settings → *Accelerator*: **GPU T4 x2** (or P100), *Internet*: **On**
   (needs a phone-verified Kaggle account).
2. *Add-ons → Secrets*: add `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`, enable both for this notebook.
3. Set `REPO_URL` below to the project's public GitHub repository.

Candles are pulled from Supabase Storage (`--sync --no-download`), so training does not
depend on the exchange API being reachable from Kaggle.

In [ ]:
REPO_URL = "https://github.com/<your-github-user>/<repo>.git"  # <- edit me
TIMESTEPS = 2_000_000

In [ ]:
%pip install -q "tradebot[ml,rl,db] @ git+{REPO_URL}"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"):
    os.environ[name] = secrets.get_secret(name)  # never print these

import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import subprocess, sys

device = "cuda" if torch.cuda.is_available() else "cpu"
cmd = ["tradebot", "rl-train", "--sync", "--no-download",
       "--timesteps", str(TIMESTEPS), "--device", device]
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-5000:], file=sys.stderr)
    raise SystemExit(result.returncode)

The last line printed above shows the new version number, whether it was promoted, and its
holdout metrics. The dashboard's **Навчання** page shows the same information.